In [1]:
import xarray as xr
import pandas as pd

ds = xr.open_zarr("../data/validation data/Caravan-Qual_lite.zarr")

target_vars = ["pH", "Fe-Dis", "Fe-Tot"]

# Load lat/lon once — both are (wqms_id,) arrays
lats = ds["wqms_lat"].compute()
lons = ds["wqms_lon"].compute()

for var in target_vars:
    if var not in ds.data_vars:
        print(f"Skipping '{var}': not found in dataset")
        continue

    da = ds[var]  # dims: (wqms_id, time)

    # Boolean mask: stations with at least one valid observation
    has_data = da.notnull().any(dim="time").compute()

    # Select matching stations
    valid_mask = has_data.values

    df = pd.DataFrame({
        "wqms_id":   ds["wqms_id"].values[valid_mask],
        "latitude":  lats.values[valid_mask],
        "longitude": lons.values[valid_mask],
    })

    filename = f"../data/validation data/stations_{var.replace('-', '_')}.csv"
    df.to_csv(filename, index=False)
    print(f"{var}: {len(df)} stations → saved to '{filename}'")

pH: 85879 stations → saved to '../data/validation data/stations_pH.csv'
Fe-Dis: 26213 stations → saved to '../data/validation data/stations_Fe_Dis.csv'
Fe-Tot: 6869 stations → saved to '../data/validation data/stations_Fe_Tot.csv'
